**Compute the Name Signals of the Development Split**

This notebook runs every candidate generator of the experiment design (`./docs/experiment_design.pdf`, Table 2) once over the development split, so that `step_5_run_development_experiments.ipynb` can compare dozens of configurations without recomputing a single embedding or phonetic code. It reads `./eval/persons_dev.csv`, `./eval/queries_dev.csv`, `./eval/d1_forms.csv` and `./results/preflight.json`. It never opens a test file, and it asserts that no test identifier appears in what it reads. It also times each generator on each query, warm and in-process, which is the query-encoding part of the latency the experiments report. It writes only to the git-ignored folder `./cache/signals`:

1. `index_forms.csv`: every distinct normalised name form of the development persons, with the one display spelling through which it is embedded
2. `emb_<model>.npz`: the embedding of every display spelling (archive forms, query tokens and dictionary forms) under E1, E2, E3a and E3b, and under E4 once per development fold
3. `dm_codes.json` and `bmpm_codes.json`: the Daitch–Mokotoff codes (computed by PostgreSQL) and Beider–Morse tokens of every normalised form
4. `trigram.npz`: the `pg_trgm` similarity of every query token to every archive form of the same script, where it is above zero
5. `query_latency.csv`: each generator's time to encode each query, in milliseconds

**1. Configuration and Helpers**

This cell defines the settings and helpers that every later cell reuses, reads the development files and builds the vocabulary every generator runs over. It asserts that every person and every person named by a query belongs to the development split, which keeps the test split sealed.

1. `index_forms`: each distinct (script, normalised form) of the development persons' tokens, with a representative display spelling: its most frequent recorded spelling, ties broken alphabetically
2. `texts_to_embed`: every display spelling an embedding model must see (archive forms, query tokens and the forms of the curated dictionary D1)
3. `forms_to_code`: every (script, normalised form) a phonetic method must code
4. `start_postgres` and `stop_postgres`: the private PostgreSQL cluster of `step_1_preflight.ipynb`

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and helpers  —  shared by every later cell
#
# Embeddings are keyed by display text, phonetic codes and trigram scores
# by normalised form. A normalised form that was recorded with several
# spellings (José, Jose) is embedded through one representative spelling,
# so that each archive form has exactly one vector per model.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import importlib.util
import json
import subprocess
import sys
import tempfile
import time
from collections import Counter, defaultdict
from pathlib import Path

import abydos.phonetic as bm_phonetic
import numpy as np
import psycopg
import torch
from abydos.phonetic._beider_morse_data import BMDATA
from anyascii import anyascii
from safetensors.numpy import load_file
from sentence_transformers import SentenceTransformer

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
EVAL_DIR        = Path("eval")                       # development inputs
PREFLIGHT_FILE  = Path("results/preflight.json")     # asset statuses
SIGNALS_DIR     = Path("cache/signals")              # every output (git-ignored)
PG_BIN          = Path(sys.executable).parent        # initdb, pg_ctl (conda env)
PG_DATA_DIR     = Path("cache/pgdata")               # the private cluster
PG_PORT         = 5433                               # socket file name only; no TCP
SEED            = 333          # torch seed for fine-tuning E4
EMBED_BATCH     = 256          # texts per batch for the PyTorch models
LATIN_LANGUAGES = "spanish,german,polish,russian,english"   # Beider–Morse, Latin input
WARM_UP_QUERIES = 20           # untimed queries before timing each generator
RULE            = "─" * 72     # frames every printed report

# E4 fine-tuning (manifest §6): InfoNCE with in-batch negatives, at most
# MAX_EPOCHS, early stopping on HELD_OUT_SHARE of the training pairs.
E4_BATCH, E4_LEARNING_RATE, E4_TEMPERATURE = 32, 2e-5, 0.07
E4_MAX_EPOCHS, E4_HELD_OUT_SHARE = 5, 0.10
EMBEDDING_MODELS = ("E1", "E2", "E3a", "E3b")


def read_csv(path: Path) -> list[dict]:
    """Read the CSV file `path` into a list of dictionaries."""
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def form_key(script: str, norm: str) -> str:
    """Return the key of a normalised form in the code files, e.g. `latin|rebeca`."""
    return f"{script}|{norm}"


def start_postgres(data_dir: Path) -> Path:
    """Start the private PostgreSQL cluster (initialised by step_1_preflight.ipynb).

    Returns:
        The socket folder, to pass to psycopg as `host`.
    """
    stop_postgres(data_dir)   # a server left running by an interrupted run
    socket_dir = Path(tempfile.mkdtemp(prefix="pg"))
    subprocess.run([PG_BIN / "pg_ctl", "-D", data_dir, "-l", data_dir / "server.log", "-w",
                    "-o", f"-k {socket_dir} -c listen_addresses= -p {PG_PORT}", "start"],
                   check=True, capture_output=True)
    return socket_dir


def stop_postgres(data_dir: Path) -> None:
    """Stop the cluster in `data_dir` if it is running."""
    subprocess.run([PG_BIN / "pg_ctl", "-D", data_dir, "-m", "fast", "stop"],
                   check=False, capture_output=True)


preflight = json.loads(PREFLIGHT_FILE.read_text(encoding="utf-8"))
dev_persons = read_csv(EVAL_DIR / "persons_dev.csv")
dev_queries = read_csv(EVAL_DIR / "queries_dev.csv")
d1_forms = read_csv(EVAL_DIR / "d1_forms.csv")

# ── The test split stays sealed: everything read here must be development ──
_dev_ids = {person["person_id"] for person in dev_persons}
assert all(person["split"] == "dev" for person in dev_persons), "a test person in persons_dev"
assert all(set(json.loads(q["relevant_person_ids"])) <= _dev_ids and q["split"] == "dev"
           for q in dev_queries), "a test person in queries_dev"

# ── Vocabulary ──────────────────────────────────────────────────────────────
_spellings = defaultdict(Counter)
for _person in dev_persons:
    for _token in json.loads(_person["tokens"]):
        _spellings[(_token["script"], _token["norm"])][_token["display"]] += 1
index_forms = [{"script": script, "norm": norm,
                "display": sorted(counts.items(), key=lambda item: (-item[1], item[0]))[0][0],
                "tokens": sum(counts.values())}
               for (script, norm), counts in sorted(_spellings.items())]
query_tokens = [token for query in dev_queries for token in json.loads(query["query_tokens"])]
texts_to_embed = sorted({form["display"] for form in index_forms}
                        | {token["display"] for token in query_tokens}
                        | {form["display"] for form in d1_forms})
forms_to_code = sorted({(form["script"], form["norm"]) for form in index_forms}
                       | {(token["script"], token["norm"]) for token in query_tokens}
                       | {(form["script"], form["norm"]) for form in d1_forms})
SIGNALS_DIR.mkdir(parents=True, exist_ok=True)
with (SIGNALS_DIR / "index_forms.csv").open("w", encoding="utf-8", newline="") as _file:
    _writer = csv.DictWriter(_file, fieldnames=list(index_forms[0]), lineterminator="\n")
    _writer.writeheader()
    _writer.writerows(index_forms)
print(f"✓ Configuration and helpers loaded ({len(dev_persons)} persons, {len(dev_queries)} "
      f"queries; {len(index_forms)} archive forms, {len(texts_to_embed)} texts to embed).")

/Users/andreiflorian/Development/cos333/cos333_gauchos_judios/search_algorithm/.conda/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Configuration and helpers loaded (507 persons, 1743 queries; 995 archive forms, 2428 texts to embed).


**2. Embeddings E1, E2, E3a and E3b**

This cell takes `texts_to_embed` and embeds every text with each pretrained model that the preflight found ready, as unit-length vectors so that a dot product is a cosine. Symphonym (E2) also takes a language code, `he` for Hebrew script and `es` for Latin script, since the Latin spellings are Spanish civil records; multilingual-e5-small (E3b) takes the `query: ` prefix its model card requires on every input. It writes, for each model:

1. `./cache/signals/emb_<model>.npz`: the arrays `texts` and `vectors`, row for row
2. a report line with the vector size and the cosine of a known cross-script pair, `Rebeca`–`רבקה`

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Embeddings E1, E2, E3a and E3b  —  one vector per display spelling
#
# E1 runs through the NumPy forward pass of the manifest's Appendix A, the
# code a production service would run. E2 loads through Symphonym's own
# inference module, with the pooling layer corrected as in step_1_preflight.
# A model the preflight marked not_run is skipped and reported.
# ─────────────────────────────────────────────────────────────────────────────
if "texts_to_embed" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


def layer_norm(x: np.ndarray, gain: np.ndarray, bias: np.ndarray) -> np.ndarray:
    """Normalise the last axis of `x`, then scale by `gain` and shift by `bias`."""
    mean = x.mean(-1, keepdims=True)
    return (x - mean) / np.sqrt(((x - mean) ** 2).mean(-1, keepdims=True) + 1e-5) * gain + bias


def e1_attention(w: dict, x: np.ndarray, prefix: str) -> np.ndarray:
    """Return the 8-head self-attention update of E1 layer `prefix` for the states `x`."""
    heads, head_dim = 8, x.shape[1] // 8
    h = layer_norm(x, w[prefix + "norm1.weight"], w[prefix + "norm1.bias"])
    qkv = h @ w[prefix + "self_attn.in_proj_weight"].T + w[prefix + "self_attn.in_proj_bias"]
    q, k, v = (t.reshape(len(x), heads, head_dim).transpose(1, 0, 2) for t in np.split(qkv, 3, -1))
    scores = np.exp(q @ k.transpose(0, 2, 1) / np.sqrt(head_dim))
    attended = ((scores / scores.sum(-1, keepdims=True)) @ v).transpose(1, 0, 2).reshape(x.shape)
    out_proj = prefix + "self_attn.out_proj."
    return attended @ w[out_proj + "weight"].T + w[out_proj + "bias"]


def e1_feed_forward(w: dict, x: np.ndarray, prefix: str) -> np.ndarray:
    """Return the ReLU feed-forward update of E1 layer `prefix` for the states `x`."""
    h = layer_norm(x, w[prefix + "norm2.weight"], w[prefix + "norm2.bias"])
    hidden = np.maximum(0, h @ w[prefix + "linear1.weight"].T + w[prefix + "linear1.bias"])
    return hidden @ w[prefix + "linear2.weight"].T + w[prefix + "linear2.bias"]


def e1_encode(w: dict, text: str) -> np.ndarray:
    """Return E1's unit-length embedding of `text`: the NumPy forward pass of the
    manifest's Appendix A, over the UTF-8 bytes, mean-pooled and normalised."""
    ids = list(text.encode("utf-8"))[:256]
    x = w["embedding.weight"][ids] + w["pos_embedding.weight"][:len(ids)]
    for layer in range(6):
        x = x + e1_attention(w, x, f"transformer.layers.{layer}.")
        x = x + e1_feed_forward(w, x, f"transformer.layers.{layer}.")
    pooled = x.mean(0)
    return pooled / np.linalg.norm(pooled)


def load_symphonym(model_dir: Path):
    """Load Symphonym v7 through its own `inference.py`, replacing its attention
    pooling by the v8 definition that matches the v7 weights (see step_1_preflight)."""

    class CorrectedAttentionPooling(torch.nn.Module):
        """Attention pooling as trained: Linear → Tanh → Dropout → Linear."""

        def __init__(self, hidden_dim: int, dropout: float = 0.1):
            """Build the two-layer scorer for states of size `hidden_dim`."""
            super().__init__()
            self.attention = torch.nn.Sequential(
                torch.nn.Linear(hidden_dim, hidden_dim // 2), torch.nn.Tanh(),
                torch.nn.Dropout(dropout), torch.nn.Linear(hidden_dim // 2, 1))

        def forward(self, x, mask=None):
            """Return the attention-weighted mean of `x` and the weights."""
            scores = self.attention(x).squeeze(-1)
            if mask is not None:
                scores = scores.masked_fill(~mask, float("-inf"))
            weights = torch.softmax(scores, dim=-1)
            return torch.bmm(weights.unsqueeze(1), x).squeeze(1), weights

    spec = importlib.util.spec_from_file_location("symphonym_inference",
                                                  model_dir / "inference.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.AttentionPooling = CorrectedAttentionPooling
    return module.SymphonymModel(model_dir=model_dir)


def symphonym_language(text: str) -> str:
    """Return E2's language code for a name: `he` for Hebrew script, else `es`."""
    return "he" if any(0x0590 <= ord(char) <= 0x05FF for char in text) else "es"


def load_encoder(model_id: str):
    """Return a function that embeds a list of texts with the model `model_id`."""
    if model_id == "E1":
        weights = load_file(preflight["E1"]["checkpoint"])
        return lambda texts: np.stack([e1_encode(weights, text) for text in texts])
    if model_id == "E2":
        symphonym = load_symphonym(Path(preflight["E2"]["model_dir"]))
        return lambda texts: symphonym.batch_embed([(t, symphonym_language(t)) for t in texts])
    prefix = "query: " if model_id == "E3b" else ""
    sentence_model = SentenceTransformer(preflight[model_id]["model"], device="cpu")
    return lambda texts: sentence_model.encode([prefix + t for t in texts],
                                               normalize_embeddings=True, batch_size=EMBED_BATCH)


def embed_all(encode, texts: list[str]) -> np.ndarray:
    """Embed `texts` in batches of EMBED_BATCH and return unit-length rows."""
    vectors = np.concatenate([np.asarray(encode(texts[start:start + EMBED_BATCH]), dtype=np.float32)
                              for start in range(0, len(texts), EMBED_BATCH)])
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


encoders = {}
print(f"{RULE}\nEmbeddings: {len(texts_to_embed)} texts\n{RULE}")
for _model_id in EMBEDDING_MODELS:
    if preflight[_model_id]["status"] != "ok":
        print(f"  !! {_model_id:<4} skipped: {preflight[_model_id]['status']}")
        continue
    _started = time.perf_counter()
    encoders[_model_id] = load_encoder(_model_id)
    _vectors = embed_all(encoders[_model_id], texts_to_embed)
    np.savez_compressed(SIGNALS_DIR / f"emb_{_model_id}.npz",
                        texts=np.array(texts_to_embed), vectors=_vectors)
    _pair = embed_all(encoders[_model_id], ["Rebeca", "רבקה"])
    print(f"  OK {_model_id:<4} {_vectors.shape[1]:>4} dimensions   cos(Rebeca, רבקה) "
          f"{float(_pair[0] @ _pair[1]):.3f}   {time.perf_counter() - _started:6.1f} s")
print(f"{RULE}\n  {len(encoders)} of {len(EMBEDDING_MODELS)} pretrained models embedded")

────────────────────────────────────────────────────────────────────────
Embeddings: 2428 texts
────────────────────────────────────────────────────────────────────────


  OK E1    256 dimensions   cos(Rebeca, רבקה) 0.889     29.2 s


  OK E2    128 dimensions   cos(Rebeca, רבקה) 0.776      0.7 s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7366.87it/s]

  OK E3a   768 dimensions   cos(Rebeca, רבקה) 0.812      8.4 s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8271.81it/s]

  OK E3b   384 dimensions   cos(Rebeca, רבקה) 0.817      5.7 s
────────────────────────────────────────────────────────────────────────
  4 of 4 pretrained models embedded


**3. E4: E1 Fine-Tuned on Archive Pairs**

This cell fine-tunes E1 once per development fold, on that fold's training pairs only, as the manifest's Experiment 1 prescribes for E4. A training pair is a Latin token and a Hebrew token of the same field recorded on the same person in the four other folds, so that the fold's own persons never inform its model. The model is E1's architecture rebuilt in PyTorch and loaded with E1's weights, trained with the InfoNCE loss of E1's authors over in-batch negatives for at most `E4_MAX_EPOCHS` epochs, keeping the weights of the epoch with the lowest loss on a held-out `E4_HELD_OUT_SHARE` of the pairs. It writes:

1. `./cache/signals/emb_E4_fold<k>.npz`: every text embedded by fold k's model
2. a report line per fold with its training pairs, the epoch kept and the held-out loss before and after

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# E4  —  E1 fine-tuned per fold on the fold's training pairs (PyTorch)
#
# The PyTorch rebuild mirrors E1's own ByteLevelEncoder: byte and position
# embeddings, six pre-norm transformer layers (8 heads, ReLU) and masked
# mean pooling. Its parameter names equal the checkpoint's, so E1's
# weights load directly, and in eval mode it reproduces e1_encode.
# ─────────────────────────────────────────────────────────────────────────────
if "encoders" not in globals():
    raise RuntimeError("Run the 'Embeddings' cell first.")


class TorchByteEncoder(torch.nn.Module):
    """E1's byte-level encoder in PyTorch, for fine-tuning."""

    def __init__(self, weights: dict):
        """Build the encoder and load the NumPy `weights` of E1's checkpoint."""
        super().__init__()
        self.embedding = torch.nn.Embedding(256, 256)
        self.pos_embedding = torch.nn.Embedding(weights["pos_embedding.weight"].shape[0], 256)
        layer = torch.nn.TransformerEncoderLayer(d_model=256, nhead=8, dim_feedforward=1024,
                                                 dropout=0.1, batch_first=True, norm_first=True)
        self.transformer = torch.nn.TransformerEncoder(layer, num_layers=6,
                                                       enable_nested_tensor=False)
        self.load_state_dict({name: torch.from_numpy(value.copy())
                              for name, value in weights.items()})

    def forward(self, texts: list[str]) -> torch.Tensor:
        """Return the unit-length embeddings of `texts`, one row per text."""
        ids = [list(text.encode("utf-8"))[:256] for text in texts]
        length = max(len(row) for row in ids)
        batch = torch.tensor([row + [0] * (length - len(row)) for row in ids])
        real = torch.tensor([[i < len(row) for i in range(length)] for row in ids])
        x = self.embedding(batch) + self.pos_embedding(torch.arange(length)).unsqueeze(0)
        x = self.transformer(x, src_key_padding_mask=~real)
        pooled = (x * real.unsqueeze(-1)).sum(1) / real.sum(1, keepdim=True)
        return torch.nn.functional.normalize(pooled, dim=-1)


def training_pairs(fold: int) -> list[tuple[str, str]]:
    """Return the (Latin, Hebrew) display pairs of the persons outside `fold`."""
    pairs = set()
    for person in dev_persons:
        if int(person["fold"]) == fold:
            continue
        tokens = json.loads(person["tokens"])
        for field_name in ("given", "surname"):
            in_field = [t for t in tokens if t["field"] == field_name]
            latin = [t["display"] for t in in_field if t["script"] == "latin"]
            hebrew = [t["display"] for t in in_field if t["script"] == "hebrew"]
            pairs.update((first, second) for first in latin for second in hebrew)
    return sorted(pairs)


def info_nce(model: TorchByteEncoder, batch: list[tuple[str, str]]) -> torch.Tensor:
    """Return the InfoNCE loss of a batch: each Latin spelling must pick out
    its own Hebrew spelling among the batch's Hebrew spellings."""
    anchors = model([latin for latin, _ in batch])
    positives = model([hebrew for _, hebrew in batch])
    logits = anchors @ positives.T / E4_TEMPERATURE
    return torch.nn.functional.cross_entropy(logits, torch.arange(len(batch)))


def held_out_loss(model: TorchByteEncoder, pairs: list[tuple[str, str]]) -> float:
    """Return the mean InfoNCE loss over `pairs`, in eval mode and without gradients."""
    model.eval()
    with torch.no_grad():
        losses = [info_nce(model, pairs[i:i + E4_BATCH]).item()
                  for i in range(0, len(pairs), E4_BATCH) if len(pairs[i:i + E4_BATCH]) > 1]
    return float(np.mean(losses))


def fine_tune(fold: int) -> tuple[TorchByteEncoder, dict]:
    """Fine-tune E1 on the training pairs of `fold`, with early stopping.

    Returns:
        The model with its best weights, and a report of the run.
    """
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    pairs = training_pairs(fold)
    order = rng.permutation(len(pairs))
    held_out = [pairs[i] for i in order[:round(len(pairs) * E4_HELD_OUT_SHARE)]]
    train = [pairs[i] for i in order[len(held_out):]]
    model = TorchByteEncoder(load_file(preflight["E1"]["checkpoint"]))
    optimiser = torch.optim.AdamW(model.parameters(), lr=E4_LEARNING_RATE)
    best = {"epoch": 0, "loss": held_out_loss(model, held_out),
            "state": {k: v.clone() for k, v in model.state_dict().items()}}
    report = {"pairs": len(pairs), "loss_before": best["loss"]}
    for epoch in range(1, E4_MAX_EPOCHS + 1):
        model.train()
        for start in range(0, len(train), E4_BATCH):
            batch = [train[i] for i in rng.permutation(len(train))[start:start + E4_BATCH]]
            if len(batch) > 1:
                optimiser.zero_grad()
                info_nce(model, batch).backward()
                optimiser.step()
        loss = held_out_loss(model, held_out)
        if loss < best["loss"]:
            best = {"epoch": epoch, "loss": loss,
                    "state": {k: v.clone() for k, v in model.state_dict().items()}}
    model.load_state_dict(best["state"])
    model.eval()
    return model, {**report, "epoch_kept": best["epoch"], "loss_after": best["loss"]}


e4_reports = {}
print(f"{RULE}\nE4: E1 fine-tuned per development fold\n{RULE}")
if preflight["E4"]["status"] != "ok":
    print(f"  !! E4 skipped: {preflight['E4']['status']}")
else:
    for _fold in sorted({int(person["fold"]) for person in dev_persons}):
        _model, e4_reports[_fold] = fine_tune(_fold)
        with torch.no_grad():
            _vectors = np.concatenate([_model(texts_to_embed[i:i + EMBED_BATCH]).numpy()
                                       for i in range(0, len(texts_to_embed), EMBED_BATCH)])
        np.savez_compressed(SIGNALS_DIR / f"emb_E4_fold{_fold}.npz",
                            texts=np.array(texts_to_embed), vectors=_vectors.astype(np.float32))
        _report = e4_reports[_fold]
        print(f"  OK fold {_fold}  {_report['pairs']:>4} pairs   epoch kept {_report['epoch_kept']}"
              f"   held-out loss {_report['loss_before']:.3f} → {_report['loss_after']:.3f}")
print(f"{RULE}\n  {len(e4_reports)} fold models fine-tuned and embedded")

────────────────────────────────────────────────────────────────────────
E4: E1 fine-tuned per development fold
────────────────────────────────────────────────────────────────────────


  OK fold 1   508 pairs   epoch kept 2   held-out loss 2.060 → 1.717


  OK fold 2   488 pairs   epoch kept 5   held-out loss 1.233 → 0.953


  OK fold 3   486 pairs   epoch kept 2   held-out loss 1.735 → 1.435


  OK fold 4   495 pairs   epoch kept 2   held-out loss 2.481 → 2.144


  OK fold 5   484 pairs   epoch kept 4   held-out loss 1.330 → 1.140
────────────────────────────────────────────────────────────────────────
  5 fold models fine-tuned and embedded


**4. Lexical Codes: Daitch–Mokotoff, Trigrams and Beider–Morse**

This cell takes `forms_to_code` and computes the lexical signals. PostgreSQL computes what the design assigns to it: the Daitch–Mokotoff codes of every form (`fuzzystrmatch`), Hebrew forms being romanised with AnyAscii first, and the `pg_trgm` similarity of every query token to every archive form of the same script. Abydos computes the Beider–Morse tokens with the Ashkenazic rules, in its approximate pass and its Hebrew-value pass, with a fixed language set for Latin input. It writes:

1. `./cache/signals/dm_codes.json`: the codes of each form, keyed `script|norm`
2. `./cache/signals/trigram.npz`: the query forms, the archive forms and the non-zero similarities between them, as a sparse matrix
3. `./cache/signals/bmpm_codes.json`: the `approx` and `hebrew` token sets of each form

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Lexical codes  —  PostgreSQL for L2 and L3, Abydos for L4
#
# The forms go into a table once; one statement then codes them all, and
# one join scores every query form against every archive form. Trigram
# similarity is computed on normalised forms, so accents and case never
# lower it.
# ─────────────────────────────────────────────────────────────────────────────
if "forms_to_code" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


def beider_morse_encoders() -> dict[str, object]:
    """Return the four Beider–Morse encoders, keyed `<script>_<pass>`, with the
    Hebrew-value pass enabled as in the manifest's Appendix B."""
    hebrew_rules = BMDATA["ash"]["hebrew"]
    for language in BMDATA["ash"]["approx"]:
        if language != "common" and language not in hebrew_rules:
            hebrew_rules[language] = ()
    bm_encoders = {}
    for script, language in (("latin", LATIN_LANGUAGES), ("hebrew", 0)):
        for match_mode in ("approx", "hebrew"):
            encoder = bm_phonetic.BeiderMorse(name_mode="ash", language_arg=language)
            encoder._match_mode = match_mode  # pylint: disable=protected-access
            bm_encoders[f"{script}_{match_mode}"] = encoder
    return bm_encoders


def postgres_codes(connection, forms: list[tuple[str, str]],
                   query_forms: list[tuple[str, str]], archive_forms: list[tuple[str, str]]):
    """Compute Daitch–Mokotoff codes and trigram similarities in PostgreSQL.

    Returns:
        The codes keyed `script|norm`, and the (query index, archive index,
        similarity) triples above zero.
    """
    dm_input = [anyascii(norm) if script == "hebrew" else norm for script, norm in forms]
    rows = connection.execute("SELECT daitch_mokotoff(x) FROM unnest(%s::text[]) AS x",
                              [dm_input]).fetchall()
    dm = {form_key(*form): sorted(set(codes or []) - {"000000"})
          for form, (codes,) in zip(forms, rows)}
    connection.execute("CREATE TEMP TABLE q (i int, script text, norm text)")
    connection.execute("CREATE TEMP TABLE f (j int, script text, norm text)")
    with connection.cursor().copy("COPY q FROM STDIN") as copy:
        for i, (script, norm) in enumerate(query_forms):
            copy.write_row((i, script, norm))
    with connection.cursor().copy("COPY f FROM STDIN") as copy:
        for j, (script, norm) in enumerate(archive_forms):
            copy.write_row((j, script, norm))
    triples = connection.execute(
        "SELECT q.i, f.j, similarity(q.norm, f.norm) FROM q JOIN f ON q.script = f.script "
        "WHERE similarity(q.norm, f.norm) > 0").fetchall()
    return dm, triples


_query_forms = sorted({(token["script"], token["norm"]) for token in query_tokens})
_archive_forms = [(form["script"], form["norm"]) for form in index_forms]
_socket_dir = start_postgres(PG_DATA_DIR)
try:
    with psycopg.connect(host=str(_socket_dir), port=PG_PORT, user="postgres",
                         dbname="postgres", autocommit=True) as _connection:
        dm_codes, _triples = postgres_codes(_connection, forms_to_code, _query_forms,
                                            _archive_forms)
finally:
    stop_postgres(PG_DATA_DIR)
_triples = np.array(_triples, dtype=np.float64).reshape(-1, 3)
np.savez_compressed(SIGNALS_DIR / "trigram.npz",
                    query_keys=np.array([form_key(*f) for f in _query_forms]),
                    archive_keys=np.array([form_key(*f) for f in _archive_forms]),
                    rows=_triples[:, 0].astype(np.int32), cols=_triples[:, 1].astype(np.int32),
                    values=_triples[:, 2].astype(np.float32))
(SIGNALS_DIR / "dm_codes.json").write_text(json.dumps(dm_codes, ensure_ascii=False),
                                           encoding="utf-8")

beider_morse = beider_morse_encoders()
bmpm_codes = {form_key(script, norm): {
    match_mode: sorted(set(beider_morse[f"{script}_{match_mode}"].encode(norm).split()))
    for match_mode in ("approx", "hebrew")} for script, norm in forms_to_code}
(SIGNALS_DIR / "bmpm_codes.json").write_text(json.dumps(bmpm_codes, ensure_ascii=False),
                                             encoding="utf-8")

print(f"{RULE}\nLexical codes: {len(forms_to_code)} forms\n{RULE}")
_example = next(key for key in sorted(dm_codes) if key.startswith("latin|") and dm_codes[key])
print(f"  OK L3 Daitch–Mokotoff   {sum(bool(c) for c in dm_codes.values()):>5} forms coded "
      f"(PostgreSQL), e.g. {_example.split('|')[1]} {dm_codes[_example]}")
print(f"  OK L2 trigrams          {len(_triples):>5} query–archive pairs above zero "
      f"({len(_query_forms)} × {len(_archive_forms)} compared)")
print(f"  OK L4 Beider–Morse      {sum(bool(c['approx']) for c in bmpm_codes.values()):>5} "
      f"forms coded (Abydos, Ashkenazic)")
print(f"{RULE}\n  lexical codes written to {SIGNALS_DIR}")

────────────────────────────────────────────────────────────────────────
Lexical codes: 2422 forms
────────────────────────────────────────────────────────────────────────
  OK L3 Daitch–Mokotoff    2419 forms coded (PostgreSQL), e.g. aan ['060000']
  OK L2 trigrams          94628 query–archive pairs above zero (1842 × 995 compared)
  OK L4 Beider–Morse       2422 forms coded (Abydos, Ashkenazic)
────────────────────────────────────────────────────────────────────────
  lexical codes written to cache/signals


**5. Query-Encoding Latency**

This cell times, for every development query and every generator, the work a production search does to encode the query before any matching: embedding its tokens for E1–E3, running the PostgreSQL lookups for L2 and L3, and computing the Beider–Morse tokens for L4. Each generator first answers `WARM_UP_QUERIES` untimed queries, so the times are warm. E4 shares E1's architecture and forward pass, so its encoding time is E1's. L1 needs no encoding beyond normalisation. It writes:

1. `./cache/signals/query_latency.csv`: `query_id, generator, ms`
2. a report of each generator's median and 95th-percentile time

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Query-encoding latency  —  warm, in-process, one query at a time
#
# The PostgreSQL lookups run the statements a production search would run
# against an indexed table of archive forms: a trigram scan of the forms
# of the query token's script, and a GIN lookup of the D–M code array.
# step_5_run_development_experiments adds the time of matching and ranking.
# ─────────────────────────────────────────────────────────────────────────────
if "dm_codes" not in globals():
    raise RuntimeError("Run the 'Lexical Codes' cell first.")


def time_queries(encode_query) -> list[float]:
    """Return the milliseconds `encode_query` takes on each development query,
    after WARM_UP_QUERIES untimed calls."""
    for query in dev_queries[:WARM_UP_QUERIES]:
        encode_query(json.loads(query["query_tokens"]))
    times = []
    for query in dev_queries:
        tokens = json.loads(query["query_tokens"])
        started = time.perf_counter()
        encode_query(tokens)
        times.append((time.perf_counter() - started) * 1000)
    return times


def postgres_lookups(connection):
    """Return the L2 and L3 query encoders that run against an indexed forms table."""
    connection.execute("CREATE TABLE IF NOT EXISTS forms (script text, norm text, dm text[])")
    connection.execute("TRUNCATE forms")
    with connection.cursor().copy("COPY forms FROM STDIN") as copy:
        for form in index_forms:
            copy.write_row((form["script"], form["norm"],
                            dm_codes.get(form_key(form["script"], form["norm"]), [])))
    connection.execute("CREATE INDEX IF NOT EXISTS forms_dm ON forms USING gin (dm)")
    connection.execute("CREATE INDEX IF NOT EXISTS forms_trgm ON forms USING gin "
                       "(norm gin_trgm_ops)")

    def trigram_lookup(tokens):
        for token in tokens:
            connection.execute("SELECT norm, similarity(norm, %s) FROM forms WHERE script = %s "
                               "AND similarity(norm, %s) > 0",
                               (token["norm"], token["script"], token["norm"])).fetchall()

    def dm_lookup(tokens):
        for token in tokens:
            text = anyascii(token["norm"]) if token["script"] == "hebrew" else token["norm"]
            connection.execute("SELECT norm FROM forms WHERE dm && daitch_mokotoff(%s)",
                               (text,)).fetchall()
    return trigram_lookup, dm_lookup


def bmpm_encode(tokens):
    """Compute the Beider–Morse tokens of every query token, in both passes."""
    for token in tokens:
        for match_mode in ("approx", "hebrew"):
            beider_morse[f"{token['script']}_{match_mode}"].encode(token["norm"])


latency = {model_id: time_queries(lambda tokens, e=encoder: e([t["display"] for t in tokens]))
           for model_id, encoder in encoders.items()}
if "E1" in latency and e4_reports:
    latency["E4"] = latency["E1"]
latency["L1"] = [0.0] * len(dev_queries)
latency["L4"] = time_queries(bmpm_encode)
_socket_dir = start_postgres(PG_DATA_DIR)
try:
    with psycopg.connect(host=str(_socket_dir), port=PG_PORT, user="postgres",
                         dbname="postgres", autocommit=True) as _connection:
        _trigram_lookup, _dm_lookup = postgres_lookups(_connection)
        latency["L2"] = time_queries(_trigram_lookup)
        latency["L3"] = time_queries(_dm_lookup)
finally:
    stop_postgres(PG_DATA_DIR)

with (SIGNALS_DIR / "query_latency.csv").open("w", encoding="utf-8", newline="") as _file:
    _writer = csv.writer(_file, lineterminator="\n")
    _writer.writerow(["query_id", "generator", "ms"])
    for _generator, _times in sorted(latency.items()):
        _writer.writerows([q["query_id"], _generator, f"{t:.3f}"]
                          for q, t in zip(dev_queries, _times))

print(f"{RULE}\nQuery-encoding latency: {len(dev_queries)} development queries\n{RULE}")
for _generator, _times in sorted(latency.items()):
    print(f"  {_generator:<4} p50 {np.percentile(_times, 50):7.2f} ms   "
          f"p95 {np.percentile(_times, 95):7.2f} ms")
print(f"{RULE}\n  {len(latency)} generators timed")

────────────────────────────────────────────────────────────────────────
Query-encoding latency: 1743 development queries
────────────────────────────────────────────────────────────────────────
  E1   p50   28.46 ms   p95   43.93 ms
  E2   p50    2.72 ms   p95    4.17 ms
  E3a  p50   44.68 ms   p95  115.82 ms
  E3b  p50   21.47 ms   p95   35.14 ms
  E4   p50   28.46 ms   p95   43.93 ms
  L1   p50    0.00 ms   p95    0.00 ms
  L2   p50    0.82 ms   p95    2.40 ms
  L3   p50    0.70 ms   p95    1.04 ms
  L4   p50    1.81 ms   p95    5.70 ms
────────────────────────────────────────────────────────────────────────
  9 generators timed


**6. Consistency Check**

This cell reads every file of `./cache/signals` back from disk and checks it against the development files, independently of the cells above:

1. every embedding file holds one unit-length vector for every text in `texts_to_embed`, and E1's vectors agree with a fresh run of the NumPy forward pass on a sample of texts; for E4, fold 1's model also embeds the sample close to E1, since fine-tuning starts from E1's weights
2. every form has a Daitch–Mokotoff entry and a Beider–Morse entry, and the trigram similarities lie in (0, 1], with every query form against itself in the archive scoring 1
3. the latency file holds one time per development query for every generator timed

It ends with the files checked and the result of the check.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the signal files against the development vocabulary
#
# Every file is re-read from disk. The E1 sample is re-encoded with the
# weights read afresh from the checkpoint, so a mismatch between the
# stored vectors and the encoder shows up here.
# ─────────────────────────────────────────────────────────────────────────────
if "latency" not in globals():
    raise RuntimeError("Run every cell above first.")

SAMPLE_TEXTS = texts_to_embed[::max(1, len(texts_to_embed) // 25)]   # about 25 texts


def embedding_problems(path: Path) -> list[str]:
    """Check one embedding file: its texts, vector norms and (E1) values."""
    stored = np.load(path)
    texts, vectors = list(stored["texts"]), stored["vectors"]
    problems = [] if texts == texts_to_embed else [f"{path.name}: texts differ"]
    norms = np.linalg.norm(vectors, axis=1)
    if not np.allclose(norms, 1, atol=1e-3):
        problems.append(f"{path.name}: {int(np.sum(~np.isclose(norms, 1, atol=1e-3)))} "
                        f"vectors are not unit-length")
    if path.name in ("emb_E1.npz", "emb_E4_fold1.npz"):
        weights, row = load_file(preflight["E1"]["checkpoint"]), dict(zip(texts, vectors))
        fresh = [float(e1_encode(weights, text) @ row[text]) for text in SAMPLE_TEXTS]
        threshold = 0.999 if path.name == "emb_E1.npz" else 0.5
        if min(fresh) < threshold:
            problems.append(f"{path.name}: cosine to a fresh E1 run falls to {min(fresh):.3f}")
    return problems


def code_problems() -> list[str]:
    """Check the code files and the trigram matrix."""
    dm = json.loads((SIGNALS_DIR / "dm_codes.json").read_text(encoding="utf-8"))
    bmpm = json.loads((SIGNALS_DIR / "bmpm_codes.json").read_text(encoding="utf-8"))
    problems = [f"{form_key(*f)}: no D–M entry" for f in forms_to_code if form_key(*f) not in dm]
    problems += [f"{form_key(*f)}: no BMPM entry" for f in forms_to_code
                 if form_key(*f) not in bmpm]
    stored = np.load(SIGNALS_DIR / "trigram.npz")
    values = stored["values"]
    if values.size and (values.min() <= 0 or values.max() > 1 + 1e-6):
        problems.append("trigram similarities outside (0, 1]")
    archive = {key: j for j, key in enumerate(stored["archive_keys"])}
    pairs = dict(zip(zip(stored["rows"], stored["cols"]), values))
    problems += [f"{key}: similarity to itself {pairs.get((i, archive[key]), 0):.2f}"
                 for i, key in enumerate(stored["query_keys"])
                 if key in archive and abs(pairs.get((i, archive[key]), 0) - 1) > 1e-6]
    return problems


def latency_problems() -> list[str]:
    """Check that every timed generator has one time per development query."""
    counts = Counter(row["generator"] for row in read_csv(SIGNALS_DIR / "query_latency.csv"))
    return [f"{generator}: {count} times for {len(dev_queries)} queries"
            for generator, count in counts.items() if count != len(dev_queries)]


_checks = {path.name: embedding_problems(path)
           for path in sorted(SIGNALS_DIR.glob("emb_*.npz"))}
_checks["codes and trigrams"] = code_problems()
_checks["query_latency.csv"] = latency_problems()
print(f"{RULE}\nConsistency: {SIGNALS_DIR} against the development vocabulary\n{RULE}")
for _name, _problems in _checks.items():
    print(f"  {'!! ' if _problems else 'OK '}{_name:<22} "
          + (f"{len(_problems)} problem(s)" if _problems else "consistent"))
    for _problem in _problems[:10]:
        print(f"        - {_problem}")
_consistent = not any(_checks.values())
print(f"{RULE}\n  {len(_checks)} signal files checked  "
      f"({'consistent' if _consistent else 'inconsistent'})")
print("\n✓ Name signals complete (embeddings, E4 per fold, lexical codes, query latency).")
print(f"  signal files consistent: {_consistent}")

────────────────────────────────────────────────────────────────────────
Consistency: cache/signals against the development vocabulary
────────────────────────────────────────────────────────────────────────
  OK emb_E1.npz             consistent
  OK emb_E2.npz             consistent
  OK emb_E3a.npz            consistent
  OK emb_E3b.npz            consistent
  OK emb_E4_fold1.npz       consistent
  OK emb_E4_fold2.npz       consistent
  OK emb_E4_fold3.npz       consistent
  OK emb_E4_fold4.npz       consistent
  OK emb_E4_fold5.npz       consistent
  OK codes and trigrams     consistent
  OK query_latency.csv      consistent
────────────────────────────────────────────────────────────────────────
  11 signal files checked  (consistent)

✓ Name signals complete (embeddings, E4 per fold, lexical codes, query latency).
  signal files consistent: True
